# Notebook
## Adnan Awad

---
### Deliverables

- Jupyter Notebook (.ipynb) with code and commentary
- Exported PDF with all figures and analysis
- Cleaned CSV

---

In [2]:
import pandas as pd
import numpy as np

# --- PART A: DATA CLEANING ---

# 1. Load the dataset
df = pd.read_csv('COVID_Country_Sample.csv')

# 2. Parse dates properly
df['date'] = pd.to_datetime(df['date'])

# 3. Handle missing values
# Justification: For COVID data, missing daily/monthly reports for cases or vaccinations
# usually imply zero reported events for that period, rather than an unknown median.
numeric_cols = ['new_cases', 'new_deaths', 'new_vaccinations', 'cases_per_million', 'vaccinations_per_hundred']
df[numeric_cols] = df[numeric_cols].fillna(0)

# 4. Handle Outliers / Spikes
# Justification: There are a few spikes for realism (often caused by
# backlogged reporting dumping all cases onto a single day/month). We will cap extreme
# outliers at the 99th percentile for each specific country to preserve the trend without
# warping the Plotly y-axis.
def cap_outliers(series):
    cap = series.quantile(0.99)
    # If the cap is 0 (e.g., no vaccinations early on), don't alter the data
    if cap == 0:
        return series
    return np.where(series > cap, cap, series)

df['new_cases'] = df.groupby('country')['new_cases'].transform(cap_outliers)
df['new_vaccinations'] = df.groupby('country')['new_vaccinations'].transform(cap_outliers)

# 5. Compute Rolling Means
# We calculate a 3-month rolling average to smooth out the reporting noise.
df = df.sort_values(by=['country', 'date'])
df['cases_rolling_avg'] = df.groupby('country')['new_cases'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['vaccines_rolling_avg'] = df.groupby('country')['new_vaccinations'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Save the cleaned dataset for the Flask app to use
df.to_csv('cleaned_covid_data.csv', index=False)
print("Data successfully cleaned and saved to 'cleaned_covid_data.csv'!")

Data successfully cleaned and saved to 'cleaned_covid_data.csv'!
